In [0]:
# =============================================================================
# Notebook: 00_utils
# Purpose: Central repository for reusable PySpark modules (functions).
#          Other notebooks will import and call these functions.
# =============================================================================
from pyspark.sql import functions as F
from datetime import datetime

def validate_schema(df, expected_columns):
    """Reusable module: Checks if the dataframe has the exact expected columns."""
    actual_columns = df.columns
    missing = set(expected_columns) - set(actual_columns)
    if missing:
        raise ValueError(f"Schema Validation Failed! Missing columns: {missing}")
    print("Reusable Module: Schema Validation Passed!")

def get_watermark(spark, watermark_path, table_name):
    """Reusable module: Fetches the last loaded timestamp for a specific table."""
    try:
        df_wm = spark.read.format("delta").load(watermark_path)
        watermark_val = df_wm.filter(F.col("table_name") == table_name).collect()[0]["last_load_timestamp"]
        print(f"Reusable Module: Watermark retrieved -> {watermark_val}")
        return watermark_val
    except Exception as e:
        # If the watermark table doesn't exist yet, return a very old date (baseline)
        print("⚠ No watermark found. Defaulting to 1900-01-01.")
        return datetime(1900, 1, 1)

def update_watermark(spark, watermark_path, table_name, new_timestamp):
    """Reusable module: Updates the control table with the new high-water mark."""
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType
    
    schema = StructType([
        StructField("table_name", StringType(), True),
        StructField("last_load_timestamp", TimestampType(), True)
    ])
    df_new_wm = spark.createDataFrame([(table_name, new_timestamp)], schema)
    
    # Overwrite the watermark (in a real scenario with many tables, we would use MERGE)
    df_new_wm.write.mode("overwrite").format("delta").save(watermark_path)
    print(f"Reusable Module: Watermark updated to -> {new_timestamp}")
